# Timing CPU and GPU inference

Load datasets for testing performance in inference

In [1]:
model_path = "Training_AdaptiveHP_acc=0.7426_ebops=1001_VU_DA_bitfile/model_Training_AdaptiveHP_acc=0.7426_ebops=1001.keras"

x_test_path = 'Data/x_test.npy'
y_test_path = 'Data/y_test.npy'

iterations = 30

In [2]:
# Check if GPU is available
import tensorflow as tf
device_name = tf.test.gpu_device_name()
if not device_name:
  compute = 'CPU'
  timings_path = 'Timings/CPU/'
  print('GPU not found')
else: 
  compute = 'GPU'
  timings_path = 'Timings/GPU/'
  print('Found GPU at: {}'.format(device_name))

2026-06-09 11:46:45.122589: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-09 11:46:45.375608: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


GPU device not found


In [4]:
import os
import sys
import time
import numpy as np
# Load input from .npy file
x_test = np.load(x_test_path)
y_test = np.load(y_test_path)

In [5]:
def test_dut():
    start = time.perf_counter()
    y_dut = model.predict(x_test, verbose=0)
    end = time.perf_counter() - start

    return [y_dut, end]

In [6]:
def cal_accuracy(y_dut):
    y_pred = np.argmax(y_dut, axis=1)
    y_true = np.argmax(y_test, axis=1)
    return np.sum(y_pred == y_true) / len(y_true)

Run the actual inference

In [7]:
from keras.models import load_model
import hgq.layers
from hgq.utils import trace_minmax

model = load_model(model_path)
# Calibrate datalane in HGQ2-model since it has layers with WRAP
trace_minmax(model, x_test, verbose=True)


dense_0: 455
dense_1: 194
dense_2: 191
dense_3: 161
Total: 1001


1001

In [8]:

timestamp = time.strftime("%Y%m%d_%H%M%S")
nr_samples = x_test.shape[0]
timings = []
for i in range(iterations):    
    # do inference
    result = test_dut()
    y_dut = result[0] 
    timings.append(result[1])

    acc = cal_accuracy(y_dut)
    print(f"\nTime: {result[1]}, Acc: {acc}")

2026-06-09 11:49:51.352755: I external/local_xla/xla/service/service.cc:163] XLA service 0x7d71d400e0e0 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
2026-06-09 11:49:51.352784: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): Host, Default Version
2026-06-09 11:49:51.496824: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1780998592.402360  503714 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
2026-06-09 11:50:04.864747: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence



Time: 20.76630156600004, Acc: 0.7435530120481928


2026-06-09 11:50:22.755077: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence



Time: 14.623059923003893, Acc: 0.7435530120481928

Time: 14.158470719004981, Acc: 0.7435530120481928


2026-06-09 11:50:51.368491: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence



Time: 14.38642038499529, Acc: 0.7435530120481928

Time: 14.018313218999538, Acc: 0.7435530120481928

Time: 14.276501967004151, Acc: 0.7435530120481928

Time: 13.882432756996423, Acc: 0.7435530120481928


2026-06-09 11:51:48.443449: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence



Time: 14.460465242002101, Acc: 0.7435530120481928

Time: 14.24406283099961, Acc: 0.7435530120481928

Time: 14.12442580700008, Acc: 0.7435530120481928

Time: 15.492352962995938, Acc: 0.7435530120481928

Time: 15.149792516000161, Acc: 0.7435530120481928

Time: 15.27978710699972, Acc: 0.7435530120481928

Time: 14.619100597999932, Acc: 0.7435530120481928

Time: 14.022954700005357, Acc: 0.7435530120481928


2026-06-09 11:53:46.964138: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence



Time: 14.786582437001925, Acc: 0.7435530120481928

Time: 14.597485774000234, Acc: 0.7435530120481928

Time: 23.597577514003206, Acc: 0.7435530120481928

Time: 22.267266333001317, Acc: 0.7435530120481928

Time: 28.82205139299913, Acc: 0.7435530120481928

Time: 14.02302656199754, Acc: 0.7435530120481928

Time: 13.857095626997761, Acc: 0.7435530120481928

Time: 12.408554287001607, Acc: 0.7435530120481928

Time: 12.455538294001599, Acc: 0.7435530120481928

Time: 12.475077996998152, Acc: 0.7435530120481928

Time: 12.436135631003708, Acc: 0.7435530120481928

Time: 12.329788771996391, Acc: 0.7435530120481928

Time: 12.40118964100111, Acc: 0.7435530120481928

Time: 12.682548611999664, Acc: 0.7435530120481928

Time: 12.794594632003282, Acc: 0.7435530120481928


In [9]:
timing_results_path = f"{timings_path}timings_dataset-{nr_samples}_{iterations}iterations_{timestamp}.txt"
np.savetxt(timing_results_path,timings)

In [10]:
# Compute statistics for collected timings (convert to ms)
import json
import numpy as np

arr = np.array(timings)
arr_ms = arr * 1e3

stats = {
    'count': int(arr_ms.size),
    'mean_ms': float(np.mean(arr_ms)),
    'median_ms': float(np.median(arr_ms)),
    'std_ms': float(np.std(arr_ms, ddof=0)),
    'min_ms': float(np.min(arr_ms)),
    'max_ms': float(np.max(arr_ms)),
    'p5_ms': float(np.percentile(arr_ms, 5)),
    'p95_ms': float(np.percentile(arr_ms, 95)),
    'inference-rate (MHz)': float((nr_samples * 1000 / np.median(arr_ms) / 1000000)),
}

# Print summary
print('Timing statistics (ms):')
for k,v in stats.items():
    print(f"{k}: {v}")

# Save JSON summary next to timings file if available
try:
    base = timing_results_path
    out = base.rsplit('.',1)[0] + '_stats.json'
except NameError:
    out = 'timing_stats.json'

with open(out, 'w') as f:
    json.dump(stats, f, indent=2)

print(f'Saved timing summary to: {out}')


Timing statistics (ms):
count: 30
mean_ms: 15181.298526867127
median_ms: 14201.266775002296
std_ms: 3695.964606119393
min_ms: 12329.788771996391
max_ms: 28822.05139299913
p5_ms: 12404.503731701334
p95_ms: 22998.937482552352
inference-rate (MHz): 0.05844549033196131
Saved timing summary to: Timings/CPU/timings_dataset-830000_30iterations_20260609_114947_stats.json


# Profile with TensorFlow
[Tensorflow profiler](https://github.com/tensorflow/tensorboard/blob/master/docs/tensorboard_profiling_keras.ipynb) 

In [ ]:
#!pip install -U tensorboard_plugin_profile

In [27]:
import tensorflow as tf
tf.profiler.experimental.start("logs/profile")

model.predict(x_test, batch_size=256)

tf.profiler.experimental.stop()


2026-06-09 11:36:14.375320: I external/local_tsl/tsl/profiler/lib/profiler_session.cc:103] Profiler session initializing.
2026-06-09 11:36:14.375341: I external/local_tsl/tsl/profiler/lib/profiler_session.cc:118] Profiler session started.


3243/3243 ━━━━━━━━━━━━━━━━━━━━ 1s 389us/step


2026-06-09 11:36:15.696351: I external/local_tsl/tsl/profiler/lib/profiler_session.cc:68] Profiler session collecting data.
2026-06-09 11:36:16.363972: I external/local_tsl/tsl/profiler/lib/profiler_session.cc:136] Profiler session tear down.
2026-06-09 11:36:16.366006: I external/local_xla/xla/tsl/profiler/rpc/client/save_profile.cc:150] Collecting XSpace to repository: logs/profile/plugins/profile/2026_06_09_11_36_16/KrissDEV.xplane.pb


In [28]:

# Load the TensorBoard notebook extension.
%load_ext tensorboard
# Launch TensorBoard and navigate to the Profile tab to view performance profile
%tensorboard --logdir=logs

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 287237), started 0:00:39 ago. (Use '!kill 287237' to kill it.)